# 10 - Report assembly

This notebook confirms the markdown report builder. We assemble the payloads with `ReportPayloadBuilder` from a lightweight stand-in `Run`, feed them plus a synthetic metrics dict and a few real figure paths into `Report.assemble`, and inspect the generated `report.md` structure (section headings, metric tables, embedded image links).

Modules exercised: `pipelines.inference_pipeline.report` (`ReportPayloadBuilder`, `Report`). Everything is written to a temporary directory.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")

import matplotlib.pyplot as plt
import numpy             as np
import torch

NP_SEED    = 0
TORCH_SEED = 0
np.random.seed(NP_SEED)
torch.manual_seed(TORCH_SEED)
torch.use_deterministic_algorithms(False)

plt.rcParams["figure.dpi"]  = 110
plt.rcParams["figure.figsize"] = (6.0, 4.0)
plt.rcParams["axes.grid"]   = True
plt.rcParams["grid.alpha"]  = 0.3

print("Repository root :", REPO_ROOT)
print("Torch device    : cpu (CPU only, no checkpoints required)")


## A minimal stand-in Run

`ReportPayloadBuilder.run_summary` reads a handful of attributes off a `Run`. Rather than load a real run, we build a tiny object exposing exactly those attributes (model name, channel counts, grid info, split region, input config). This keeps the notebook free of the data mount while matching the accessed interface.

In [ ]:
from types import SimpleNamespace
from pipelines.inference_pipeline.report import ReportPayloadBuilder, Report
from pipelines.dataset_pipeline.spatial import Patcher

grid = Patcher.build(spatial_size=(40, 40), patch_size=(16, 16), stride=8).grid

region = SimpleNamespace(as_tuple=lambda: (0, 40, 0, 40))
crop   = SimpleNamespace(as_tuple=lambda: (0, 64, 0, 64))
input_cfg = SimpleNamespace(as_dict=lambda: {'use_amplitude': True, 'use_phase': True})
dataset_cfg = SimpleNamespace(
    preprocessing_run_directory=Path('/tmp/preprocess_demo'),
    input_config=input_cfg, batch_size=8,
)

run = SimpleNamespace(
    model_name='unet', in_channels=4, out_channels=6, n_gaussians=2,
    x_axis_length=64, split_name='test',
    split_region=region, global_crop=crop, grid=grid,
    dataset_config=dataset_cfg, used_ema=True,
    checkpoint_meta={'epoch': 120, 'best_epoch': 98, 'best_val_loss': 0.0123},
)
print('stand-in Run ready:', run.model_name, run.in_channels, '->', run.out_channels)


## Build the report payloads

`run_summary` and `inference_config` translate the run and config into the dicts the report consumes. We pass a real `InferenceConfig` for the config payload.

In [ ]:
from configuration.inference_config import InferenceConfig

x_axis_np = np.linspace(-32.0, 32.0, run.x_axis_length).astype(np.float64)
cfg       = InferenceConfig(run_directory=Path('/tmp/run_demo'))

run_summary_payload = ReportPayloadBuilder.run_summary(run, x_axis_np)
infer_cfg_payload   = ReportPayloadBuilder.inference_config(cfg, run)

print('run_summary keys     :', sorted(run_summary_payload)[:6], '...')
print('inference_config keys:', sorted(infer_cfg_payload)[:6], '...')


## A synthetic metrics dict and real figure files

We provide a small headline-metrics dict (the report only reads keys it knows about, missing keys render as a dash) and create two trivial PNG figures so the embedded-image section has real, relative-linkable paths.

In [ ]:
import tempfile

out_dir = Path(tempfile.mkdtemp(prefix='nb10_'))
fig_dir = out_dir / 'figures' / 'profiles'
fig_dir.mkdir(parents=True, exist_ok=True)

fig_paths = []
for i in range(2):
    fig, ax = plt.subplots(figsize=(3.0, 2.0))
    ax.plot(x_axis_np, np.exp(-((x_axis_np - 4 * i) ** 2) / 50.0))
    ax.set_title(f'demo profile {i}')
    p = fig_dir / f'demo_{i}.png'
    fig.savefig(p, dpi=90); plt.close(fig)
    fig_paths.append(p)

global_metrics = {
    'curve_mse_gt': 0.0021, 'curve_mae_gt': 0.031, 'curve_rmse_gt': 0.046,
    'overall_r2_gt': 0.94, 'psnr_db_gt': 27.3,
    'pixel_mse_gt_mean': 0.0025, 'pixel_mse_gt_median': 0.0019, 'pixel_mse_gt_p95': 0.006,
    'pixel_r2_gt_mean': 0.92, 'pixel_r2_gt_median': 0.95,
    'ssim_gt_elev_mean': 0.88, 'ssim_gt_range_mean': 0.81, 'ssim_gt_azimuth_mean': 0.83,
    'gt_mean': 0.12, 'gt_std': 0.20, 'n_pixels': 1600, 'n_elevation': 64,
    'x_axis_min': float(x_axis_np.min()), 'x_axis_max': float(x_axis_np.max()),
}
print('figures:', [p.name for p in fig_paths])


## Assemble the report

`Report.assemble` writes `report.md` and returns its path. The figure-path dict is keyed by section name; we only populate the profiles group here.

In [ ]:
report = Report(
    output_dir=out_dir,
    run_summary=run_summary_payload,
    inference_config=infer_cfg_payload,
    checkpoint_meta=run.checkpoint_meta,
    global_metrics=global_metrics,
    figure_paths={'profiles_best': fig_paths},
    gif_paths={},
    report_path=out_dir / 'report.md',
)
report_path = report.assemble()
text = report_path.read_text(encoding='utf-8')
print('report bytes:', len(text))


## Inspect the generated structure

We list the section headings and check that the headline metrics and a relative image link landed in the document.

In [ ]:
headings = [ln for ln in text.splitlines() if ln.startswith('#')]
print('Headings')
for h in headings:
    print('   ', h)

print()
print('contains MSE value 0.0021 :', '0.0021' in text)
print('contains relative img link:', '![' in text and 'figures/profiles/demo_0.png' in text)


## Preview a slice of the markdown

Print the headline-metrics section so the rendered tables can be eyeballed.

In [ ]:
lines = text.splitlines()
start = next(i for i, ln in enumerate(lines) if ln.startswith('## 2.'))
end   = next(i for i, ln in enumerate(lines[start + 1:], start + 1) if ln.startswith('## 3.'))
print('\n'.join(lines[start:end]))


## Expected visual outcome

The assembled report must contain the numbered sections (run summary, headline metrics, full metric tables, figures), the headline MSE value `0.0021` must appear in a table, and the profiles section must embed a relative image link to `figures/profiles/demo_0.png`. The printed headline section should show well-formed markdown tables, confirming the report builder wires payloads, metrics and figure paths into a coherent document.